In [2]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import xy
import os

print("=" * 60)
print("EXTRACTION DES COORDONNÉES - TOUS LES FICHIERS")
print("=" * 60)

# 1. Charger X.npy et y.npy
base_path = r'C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\processed\california_preprocessed'

X = np.load(os.path.join(base_path, 'X.npy'))
y = np.load(os.path.join(base_path, 'y.npy'))

n_target = X.shape[0]
print(f"\n📂 Données Partie 1: {n_target} points cibles")

# 2. Tous vos fichiers GeoTIFF
tif_files = [
    # North California - 400km² (D:)
    r'D:\datasentinel\MCTNet_400km2_California_North-1.tif',
    r'D:\datasentinel\MCTNet_400km2_California_North-2.tif',
    r'D:\datasentinel\MCTNet_400km2_California_North-3.tif',
    r'D:\datasentinel\MCTNet_400km2_California_North-4.tif',
    # South California - 400km² (D:)
    r'D:\datasentinel\MCTNet_400km2_California_South-1.tif',
    r'D:\datasentinel\MCTNet_400km2_California_South-2.tif',
    r'D:\datasentinel\MCTNet_400km2_California_South-3.tif',
    r'D:\datasentinel\MCTNet_400km2_California_South-4.tif',
    # North California - 14GB (C:)
    r'C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\sentinel2\MCTNet_14GB_California_North-5.tif',
    r'C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\data\raw\california\sentinel2\MCTNet_14GB_California_North-6.tif',
]

# 3. Vérifier quels fichiers existent
valid_files = []
for f in tif_files:
    if os.path.exists(f):
        valid_files.append(f)
        # Obtenir la taille du fichier
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"✅ {os.path.basename(f)} ({size_mb:.1f} MB)")
    else:
        print(f"❌ {os.path.basename(f)} (non trouvé)")

print(f"\n📁 Fichiers valides: {len(valid_files)}/{len(tif_files)}")

if len(valid_files) == 0:
    print("\n❌ Aucun fichier trouvé! Vérifiez les chemins.")
    exit()

# 4. Extraire les coordonnées de tous les fichiers
print(f"\n📍 Extraction des coordonnées...")

all_coords = []
total_pixels = 0

for file_path in valid_files:
    print(f"\n   Traitement: {os.path.basename(file_path)}")
    
    with rasterio.open(file_path) as src:
        height, width = src.height, src.width
        transform = src.transform
        
        n_pixels = width * height
        total_pixels += n_pixels
        print(f"      Dimensions: {width} x {height} = {n_pixels:,} pixels")
        
        # Extraire toutes les coordonnées
        coords_count = 0
        for row in range(height):
            for col in range(width):
                lon, lat = xy(transform, row, col)
                all_coords.append({
                    'lat': lat,
                    'lon': lon,
                    'file': os.path.basename(file_path)
                })
                coords_count += 1
        
        print(f"      Extraits: {coords_count:,} points")

print(f"\n📊 Total pixels extraits: {len(all_coords):,}")
print(f"   Objectif: {n_target:,} points")

# 5. Sous-échantillonner pour correspondre exactement à X
if len(all_coords) > n_target:
    step = len(all_coords) // n_target
    df_coords = pd.DataFrame(all_coords).iloc[::step][:n_target].reset_index(drop=True)
    print(f"   Échantillonnage: 1 pixel sur {step}")
elif len(all_coords) < n_target:
    repeats = (n_target // len(all_coords)) + 1
    df_coords = pd.concat([pd.DataFrame(all_coords)] * repeats).iloc[:n_target].reset_index(drop=True)
    print(f"   Répétition pour atteindre {n_target} points")
else:
    df_coords = pd.DataFrame(all_coords)
    print(f"   Parfait: nombre exact de points")

print(f"   Points finaux: {len(df_coords):,}")

# 6. Ajouter les labels
df_coords['label'] = y[:len(df_coords)]

# 7. Sauvegarder
output_path = r'C:\Users\KM-USER\Documents\M1\SII\S2\ResNeur\projectCropClassification\part2\data'
os.makedirs(output_path, exist_ok=True)

csv_path = os.path.join(output_path, 'all_pixels_coords.csv')
df_coords.to_csv(csv_path, index=False)

print(f"\n✅ Fichier sauvegardé: {csv_path}")
print(f"   Shape final: {df_coords.shape}")

# 8. Aperçu
print("\n📋 Aperçu des 5 premières lignes:")
print(df_coords.head())

print("\n📊 Statistiques des coordonnées:")
print(f"   Latitude: min={df_coords['lat'].min():.4f}, max={df_coords['lat'].max():.4f}")
print(f"   Longitude: min={df_coords['lon'].min():.4f}, max={df_coords['lon'].max():.4f}")

# 9. Distribution par fichier
print("\n📊 Distribution par fichier source:")
print(df_coords['file'].value_counts())

print("\n📊 Distribution des labels:")
label_counts = df_coords['label'].value_counts().sort_index()
for label, count in label_counts.items():
    print(f"   Label {label}: {count} points")

print("\n✅ Extraction terminée !")

EXTRACTION DES COORDONNÉES - TOUS LES FICHIERS

📂 Données Partie 1: 10487 points cibles
✅ MCTNet_400km2_California_North-1.tif (3167.0 MB)
✅ MCTNet_400km2_California_North-2.tif (1299.1 MB)
✅ MCTNet_400km2_California_North-3.tif (376.4 MB)
✅ MCTNet_400km2_California_North-4.tif (152.5 MB)
✅ MCTNet_400km2_California_South-1.tif (3509.7 MB)
✅ MCTNet_400km2_California_South-2.tif (1353.9 MB)
✅ MCTNet_400km2_California_South-3.tif (419.0 MB)
✅ MCTNet_400km2_California_South-4.tif (160.5 MB)
✅ MCTNet_14GB_California_North-5.tif (3662.3 MB)
✅ MCTNet_14GB_California_North-6.tif (729.0 MB)

📁 Fichiers valides: 10/10

📍 Extraction des coordonnées...

   Traitement: MCTNet_400km2_California_North-1.tif
      Dimensions: 1792 x 1792 = 3,211,264 pixels
      Extraits: 3,211,264 points

   Traitement: MCTNet_400km2_California_North-2.tif
      Dimensions: 752 x 1792 = 1,347,584 pixels
      Extraits: 1,347,584 points

   Traitement: MCTNet_400km2_California_North-3.tif
      Dimensions: 1792 x 212 